In [122]:
# import sys
# !{sys.executable} -m pip uninstall prettytable -y
# !{sys.executable} -m pip uninstall ipython-sql -y
# !{sys.executable} -m pip install prettytable==3.9.0
# !{sys.executable} -m pip install ipython-sql==0.4.1

In [123]:
# !pip uninstall sqlalchemy -y
# !pip install sqlalchemy==1.4.49

In [124]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [125]:
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False

In [126]:
import sys
print(sys.executable)

f:\Python\FastAPI01\myenv\Scripts\python.exe


In [127]:
import os

In [128]:
host = os.getenv('SQL_HOST', 'localhost')
port = os.getenv('SQL_PORT', '5432')
user = os.getenv('SQL_USER')
password = os.getenv('SQL_PASSWORD')
database = os.getenv('SQL_DB', 'sdb')

In [129]:
connection_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"

In [130]:
from sqlalchemy import create_engine

engine = create_engine(connection_string)

In [131]:

%sql $connection_string

In [132]:
%%sql
SELECT * FROM nyc_neighborhoods WHERE FALSE

0 rows affected.


""


In [133]:
%%sql 
SELECT * FROM nyc_neighborhoods LIMIT 5;

5 rows affected.


,id,geom,boroname,name
0,1,0106000020266900000100000001030000000100000011...,Brooklyn,Bensonhurst
1,2,0106000020266900000100000001030000000100000008...,Manhattan,East Village
2,3,0106000020266900000100000001030000000100000034...,Manhattan,West Village
3,4,0106000020266900000100000001030000000100000057...,The Bronx,Throggs Neck
4,5,0106000020266900000100000001030000000100000066...,The Bronx,Wakefield-Williamsbridge


In [134]:
%%sql
SELECT * FROM nyc_neighborhoods WHERE boroname LIKE '%Manhattan%';

28 rows affected.


,id,geom,boroname,name
0,2,0106000020266900000100000001030000000100000008...,Manhattan,East Village
1,3,0106000020266900000100000001030000000100000034...,Manhattan,West Village
2,7,010600002026690000010000000103000000010000001F...,Manhattan,Battery Park
3,8,0106000020266900000100000001030000000100000006...,Manhattan,Carnegie Hill
4,11,0106000020266900000100000001030000000100000017...,Manhattan,Harlem
5,12,0106000020266900000100000001030000000100000016...,Manhattan,Gramercy
6,22,0106000020266900000100000001030000000100000012...,Manhattan,Soho
7,30,010600002026690000010000000103000000010000000C...,Manhattan,Morningside Heights
8,31,0106000020266900000100000001030000000100000007...,Manhattan,Murray Hill
9,48,010600002026690000010000000103000000010000000D...,Manhattan,Central Park


In [135]:
%%sql
SELECT avg(char_length(name)), stddev(char_length(name)) 
FROM nyc_neighborhoods
WHERE boroname = 'Manhattan';

1 rows affected.


,avg,stddev
0,11.8214285714285714,4.3123729948325257


In [136]:
%%sql
SELECT boroname, avg(char_length(name)), stddev(char_length(name)) 
FROM nyc_neighborhoods
GROUP BY boroname;

5 rows affected.


,boroname,avg,stddev
0,Queens,11.6666666666666667,5.0057438272815975
1,Brooklyn,11.7391304347826087,3.9105613559407395
2,Staten Island,12.2916666666666667,5.2043390480959474
3,The Bronx,12.0416666666666667,3.6651017740975152
4,Manhattan,11.8214285714285714,4.3123729948325257


In [137]:
%%sql
SELECT * FROM nyc_census_blocks LIMIT 5;

5 rows affected.


,id,geom,blkid,popn_total,popn_white,popn_black,popn_nativ,popn_asian,popn_other,boroname
0,1,0106000000010000000103000000010000000A00000051...,360850009001000,97,51,32,1,5,8,Staten Island
1,2,0106000000010000000103000000010000000700000008...,360850020011000,66,52,2,0,7,5,Staten Island
2,3,0106000000010000000103000000010000000600000082...,360850040001000,62,14,18,2,25,3,Staten Island
3,4,0106000000010000000103000000010000000A00000011...,360850074001000,137,92,12,0,13,20,Staten Island
4,5,01060000000100000001030000000100000014000000EA...,360850096011000,289,230,0,0,32,27,Staten Island


In [138]:
%%sql
SELECT Sum(popn_total) AS total_population
FROM nyc_census_blocks
WHERE boroname = 'Manhattan';

1 rows affected.


,total_population
0,1585873


In [139]:
%%sql
SELECT DISTINCT boroname, Sum(popn_total) AS total_population
FROM nyc_census_blocks
GROUP BY boroname
ORDER BY total_population DESC;

5 rows affected.


,boroname,total_population
0,Brooklyn,2504700
1,Queens,2230621
2,Manhattan,1585873
3,The Bronx,1385108
4,Staten Island,468730


In [140]:
%%sql
SELECT boroname,
100*Sum(popn_white)/Sum(popn_total) AS pct_white,
100*Sum(popn_black)/Sum(popn_total) AS pct_black,
100*Sum(popn_asian)/Sum(popn_total) AS pct_asian,
100*Sum(popn_other)/Sum(popn_total) AS pct_other
FROM nyc_census_blocks
GROUP BY boroname;

5 rows affected.


,boroname,pct_white,pct_black,pct_asian,pct_other
0,Queens,39.7220773945910130,19.1253467083830019,22.9428486506672357,17.5144500119025150
1,Brooklyn,42.8011737932686549,34.3387631253243901,10.4713538547530642,11.8487643230726235
2,The Bronx,27.9037446899447552,36.4736901382419277,3.5815979692558270,30.7226584497382154
3,Manhattan,57.4493039480462811,15.5552809083703424,11.3219658825139214,15.1268102805205713
4,Staten Island,72.8942034860154033,10.6366138288566979,7.5019734175324814,8.6055938386704499


In [141]:
%%sql
CREATE EXTENSION IF NOT EXISTS postgis;
DROP TABLE IF EXISTS geometrics;

CREATE TABLE geometrics (
    name varchar,
    geom geometry
);

INSERT INTO geometrics VALUES
('point', ST_GeomFromText('POINT(1 2)', 4326)),
('line', ST_GeomFromText('LINESTRING(0 0, 1 1, 2 2)', 4326)),
('polygon', ST_GeomFromText('POLYGON((0 0, 0 1, 1 1, 1 0, 0 0))', 4326)),
('polygon with hole', ST_GeomFromText('POLYGON((0 0, 0 2, 2 2, 2 0, 0 0), (0.5 0.5, 0.5 1.5, 1.5 1.5, 1.5 0.5, 0.5 0.5))', 4326)),
('collection', ST_GeomFromText('GEOMETRYCOLLECTION(POINT(1 2), LINESTRING(0 0, 1 1, 2 2), POLYGON((0 0, 0 1, 1 1, 1 0, 0 0)))', 4326));

SELECT name, ST_AsText(geom) FROM geometrics;


Done.
Done.
Done.
5 rows affected.
5 rows affected.


,name,st_astext
0,point,POINT(1 2)
1,line,"LINESTRING(0 0,1 1,2 2)"
2,polygon,"POLYGON((0 0,0 1,1 1,1 0,0 0))"
3,polygon with hole,"POLYGON((0 0,0 2,2 2,2 0,0 0),(0.5 0.5,0.5 1.5..."
4,collection,"GEOMETRYCOLLECTION(POINT(1 2),LINESTRING(0 0,1..."


to see all the data tables in database

In [142]:
%%sql
SELECT * FROM geometry_columns;

6 rows affected.


,f_table_catalog,f_table_schema,f_table_name,f_geometry_column,coord_dimension,srid,type
0,sdb,public,nyc_subway_stations,geom,2,26918,POINT
1,sdb,public,nyc_homicides,geom,2,0,POINT
2,sdb,public,nyc_streets,geom,2,26918,MULTILINESTRING
3,sdb,public,nyc_census_blocks,geom,2,0,MULTIPOLYGON
4,sdb,public,nyc_neighborhoods,geom,2,26918,MULTIPOLYGON
5,sdb,public,geometrics,geom,2,0,GEOMETRY


To see or drop a saved query as view, use the term 'view' in place of table

In [143]:
%%sql
DROP VIEW "HIGH_ASIAN"

(psycopg2.errors.UndefinedTable) view "HIGH_ASIAN" does not exist

[SQL: DROP VIEW "HIGH_ASIAN"]
(Background on this error at: https://sqlalche.me/e/14/f405)


wkt = well known type, ST = spatial type

In [144]:
%%sql
SELECT ST_AsText(geom) AS wkt, ST_SRID(geom) AS srid, ST_NDims(geom) AS ndims, ST_NPoints(geom) AS npoints
FROM geometrics;

5 rows affected.


,wkt,srid,ndims,npoints
0,POINT(1 2),4326,2,1
1,"LINESTRING(0 0,1 1,2 2)",4326,2,3
2,"POLYGON((0 0,0 1,1 1,1 0,0 0))",4326,2,5
3,"POLYGON((0 0,0 2,2 2,2 0,0 0),(0.5 0.5,0.5 1.5...",4326,2,10
4,"GEOMETRYCOLLECTION(POINT(1 2),LINESTRING(0 0,1...",4326,2,9


ST_X(col_name) returns the x-coordinate, and ST_Y(col_name) returns the y-coordinate

In [145]:
%%sql
SELECT ST_X(geom) x, ST_Y(geom) y
FROM geometrics
WHERE name = 'point';

1 rows affected.


,x,y
0,1.0,2.0


In [146]:
%%sql
SELECT boroname, ST_AsText(geom) FROM nyc_census_blocks LIMIT 10;

10 rows affected.


,boroname,st_astext
0,Staten Island,MULTIPOLYGON(((577856.5470479821 4499583.23492...
1,Staten Island,MULTIPOLYGON(((578620.7173632095 4495974.81786...
2,Staten Island,MULTIPOLYGON(((577227.2244709881 4495995.06684...
3,Staten Island,MULTIPOLYGON(((579037.0332016965 4494421.76981...
4,Staten Island,MULTIPOLYGON(((577652.4825280879 4494975.05228...
5,Staten Island,MULTIPOLYGON(((577305.0606213677 4492950.12053...
6,Staten Island,MULTIPOLYGON(((575441.898302898 4491397.132737...
7,Staten Island,MULTIPOLYGON(((573055.2945856147 4491621.86005...
8,Staten Island,MULTIPOLYGON(((571826.8104344463 4490858.12029...
9,Staten Island,MULTIPOLYGON(((570905.0397259424 4490255.70022...


In [147]:
%%sql
SELECT name, ST_AsText(geom), ST_Length(geom) AS length, ST_AsText(ST_StartPoint(geom)) AS start, ST_AsText(ST_EndPoint(geom)) AS end
FROM nyc_streets
ORDER BY ST_Length(geom) DESC LIMIT 100;

100 rows affected.


,name,st_astext,length,start,end
0,Leif Ericson Dr,MULTILINESTRING((582719.4277739702 4499363.982...,14918.122617,POINT(582719.4277739702 4499363.982950422),None
1,Hylan Blvd,MULTILINESTRING((578523.3792354487 4494605.949...,14010.051237,POINT(578523.3792354487 4494605.949495728),None
2,Grand Central Pky,MULTILINESTRING((608629.5450946327 4512863.763...,14004.747130,POINT(608629.5450946327 4512863.76363684),None
3,Nostrand Ave,MULTILINESTRING((588422.61620682 4505915.70696...,12916.485763,POINT(588422.61620682 4505915.706966937),None
4,Ocean Ave,MULTILINESTRING((588266.4540993671 4497756.269...,12342.720041,POINT(588266.4540993671 4497756.269439748),None
...,...,...,...,...,...
95,Richmond Ter,MULTILINESTRING((578192.6774662302 4499445.793...,5298.372570,POINT(578192.6774662302 4499445.7936787605),None
96,Hutchinson River Pky,MULTILINESTRING((598491.0315048957 4523571.853...,5288.602010,POINT(598491.0315048957 4523571.85332119),None
97,18th Ave,MULTILINESTRING((587301.773982865 4498778.5451...,5266.690631,POINT(587301.773982865 4498778.54514366),None
98,Sutter Ave,MULTILINESTRING((591022.422070461 4502150.7022...,5258.397783,POINT(591022.422070461 4502150.702293016),None


In [148]:
%%sql
SELECT n.name,
       MIN(ST_Distance(n.geom, s.geom)) AS nearest_distance
FROM nyc_neighborhoods n
JOIN nyc_subway_stations s
ON s.color = 'RED'
GROUP BY n.name;

129 rows affected.


,name,nearest_distance
0,Ardon Heights,20335.751523
1,Astoria-Long Island City,2655.724629
2,Oakwood,15794.234999
3,High Bridge,355.613556
4,Yorkville,331.412251
...,...,...
124,Tremont,1447.409674
125,Parkchester,4684.758464
126,East Brooklyn,7580.479686
127,Woodhaven-Richmond Hill,9793.210017


Now working with polygons

In [149]:
%%sql
SELECT name, ST_Area(geom) AS area
FROM geometrics
WHERE name IN ('polygon', 'polygon with hole', 'collection');

3 rows affected.


,name,area
0,polygon,1.0
1,polygon with hole,3.0
2,collection,1.0


In [150]:
%%sql
SELECT ST_Area(geom) AS area, ST_Perimeter(geom) AS perimeter, ST_NRings(geom) AS nrings, ST_InteriorRingN(geom, 1) AS interior_ring, ST_ExteriorRing(geom) AS exterior_ring
FROM geometrics
WHERE name IN ('polygon', 'polygon with hole');

2 rows affected.


,area,perimeter,nrings,interior_ring,exterior_ring
0,1.0,4.0,1,None,0102000020E61000000500000000000000000000000000...
1,3.0,12.0,2,0102000020E610000005000000000000000000E03F0000...,0102000020E61000000500000000000000000000000000...


In [151]:
%%sql
--Postgres accepts many transformations, like ST_AsGeoJSON(), ST_AsEWKT(), ST_AsSVG(), etc. 

SELECT name, ST_AsGeoJSON(geom) AS geojson, ST_AsEWKT(geom) AS ewkt, ST_AsSVG(geom) AS svg
FROM geometrics;

5 rows affected.


,name,geojson,ewkt,svg
0,point,"{""type"":""Point"",""coordinates"":[1,2]}",SRID=4326;POINT(1 2),"cx=""1"" cy=""-2"""
1,line,"{""type"":""LineString"",""coordinates"":[[0,0],[1,1...","SRID=4326;LINESTRING(0 0,1 1,2 2)",M 0 0 L 1 -1 2 -2
2,polygon,"{""type"":""Polygon"",""coordinates"":[[[0,0],[0,1],...","SRID=4326;POLYGON((0 0,0 1,1 1,1 0,0 0))",M 0 0 L 0 -1 1 -1 1 0 Z
3,polygon with hole,"{""type"":""Polygon"",""coordinates"":[[[0,0],[0,2],...","SRID=4326;POLYGON((0 0,0 2,2 2,2 0,0 0),(0.5 0...",M 0 0 L 0 -2 2 -2 2 0 Z M 0.5 -0.5 L 0.5 -1.5 ...
4,collection,"{""type"":""GeometryCollection"",""geometries"":[{""t...","SRID=4326;GEOMETRYCOLLECTION(POINT(1 2),LINEST...","cx=""1"" cy=""-2"";M 0 0 L 1 -1 2 -2;M 0 0 L 0 -1 ..."


In [152]:
%%sql
--casting from geometry to text is implicit in many cases, but you can also do it explicitly with ST_AsText() or ::text
SELECT name, geom::text AS wkt
FROM geometrics;

5 rows affected.


,name,wkt
0,point,0101000020E6100000000000000000F03F000000000000...
1,line,0102000020E61000000300000000000000000000000000...
2,polygon,0103000020E61000000100000005000000000000000000...
3,polygon with hole,0103000020E61000000200000005000000000000000000...
4,collection,0107000020E61000000300000001010000000000000000...


In [153]:
%%sql
SELECT 25::float / 4 AS result;

1 rows affected.


,result
0,6.25


In [154]:
%%sql
SELECT 'SRID=4326;POINT(1 2)'::geometry AS geom;

1 rows affected.


,geom
0,0101000020E6100000000000000000F03F000000000000...


Spatial relationships

In [155]:
%%sql
SELECT * FROM nyc_subway_stations LIMIT 5;

5 rows affected.


,id,geom,objectid,name,alt_name,cross_st,long_name,label,borough,nghbhd,routes,transfers,color,express,closed
0,376,010100002026690000371775B5C3CE2141CBD234777131...,1,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,None
1,2,010100002026690000CBE327F938CD21415EDBE1572D31...,2,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,None
2,1,010100002026690000C676635D10CD2141A0ECDB697530...,3,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,None
3,125,010100002026690000F4CF3E3654032241B5704681A73C...,4,138th St,Grand Concourse,Grand Concourse,"138th St / Grand Concourse (4,5) Bronx","138th St / Grand Concourse (4,5)",Bronx,None,"4,5","4,5",GREEN,None,None
4,126,01010000202669000084DADF7AED0422410C380E6E3A3D...,5,149th St,Grand Concourse,Grand Concourse,149th St / Grand Concourse (4) Bronx,149th St / Grand Concourse (4),Bronx,None,4,"2,4,5",GREEN,express,None


In [168]:
%config SqlMagic.displaylimit = None

In [172]:
%%sql
SELECT name, geom, ST_asText(geom) AS wkt
FROM nyc_subway_stations
WHERE name = 'South Ferry';

1 rows affected.


,name,geom,wkt
0,South Ferry,010100002026690000C676635D10CD2141A0ECDB697530...,POINT(583304.1823994748 4506069.654048115)


In [175]:
%%sql
SELECT name
FROM nyc_subway_stations
WHERE ST_Equals(geom, ST_GeomFromText('POINT(583304.1823994748 4506069.654048115)', 26918));

1 rows affected.


,name
0,South Ferry


In [176]:
%%sql
SELECT name, ST_AsText(geom)
FROM nyc_subway_stations
WHERE name = 'South Ferry';

1 rows affected.


,name,st_astext
0,South Ferry,POINT(583304.1823994748 4506069.654048115)


In [177]:
%%sql
select name, boroname
from nyc_neighborhoods
where ST_Intersects(geom, ST_GeomFromText('POINT(583304.1823994748 4506069.654048115)', 26918));

1 rows affected.


,name,boroname
0,Financial District,Manhattan


In [178]:
%%sql
select name, boroname
from nyc_neighborhoods
where ST_Disjoint(geom, ST_GeomFromText('POINT(583304.1823994748 4506069.654048115)', 26918));

128 rows affected.


,name,boroname
0,Bensonhurst,Brooklyn
1,East Village,Manhattan
2,West Village,Manhattan
3,Throggs Neck,The Bronx
4,Wakefield-Williamsbridge,The Bronx
...,...,...
123,Red Hook,Brooklyn
124,Douglastown-Little Neck,Queens
125,Whitestone,Queens
126,Steinway,Queens


In [179]:
%%sql
SELECT ST_Distance(
    ST_GeomFromText('POINT(583304.1823994748 4506069.654048115)', 26918),
    ST_GeomFromText('POINT(583500 4506000)', 26918)
) AS distance;

1 rows affected.


,distance
0,207.837001


In [182]:
%%sql
SELECT name
FROM nyc_streets
WHERE ST_Dwithin(geom, ST_GeomFromText('POINT(583304.1823994748 4506069.654048115)', 26918), 1000);

94 rows affected.


,name
0,Ferry Line Rd
1,Carder Rd
2,1st Pl
3,Andes Rd
4,2nd Pl
...,...
89,Cliff St
90,De Peyster St
91,FDR Dr
92,FDR Dr
